In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 03. Multi-Band Transmission (O+E+S+C+L)\n",
    "Scaling up to the full 42.4 THz O+E+S+C+L band plan (1,273 channels) over 80 km SSMF.\n",
    "\n",
    "**Goal:** Demonstrate the massive NLI penalty and Raman cross-talk introduced when expanding from single-band to ultra-wideband transmission. This establishes the 'problem' that the neural model will solve."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('..')\n",
    "import yaml\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from src.modulation import build_band_plan, generate_pcs_symbols\n",
    "from src.fiber_channel import run_ssfm\n",
    "from src.amplifiers import apply_amplifiers\n",
    "from src.dsp import receiver_dsp\n",
    "from src.metrics import compute_metrics\n",
    "\n",
    "with open('../config/simulation_params.yaml', 'r') as f:\n",
    "    cfg = yaml.safe_load(f)\n",
    "\n",
    "band_plan = build_band_plan(cfg['bands'], cfg['aggregate'])\n",
    "print(f\"Total channels: {band_plan['n_channels']}\")\n",
    "print(f\"Total bandwidth: {cfg['aggregate']['total_bandwidth_THz']} THz\")\n",
    "\n",
    "# Generate Uniform QAM (No PCS yet)\n",
    "mod_cfg = cfg['modulation'].copy()\n",
    "mod_cfg['pcs_learning_method'] = 'none'\n",
    "\n",
    "tx = generate_pcs_symbols(band_plan, mod_cfg, cfg['launch_power'], samples=16384)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Run Full Pipeline\n",
    "rx_raw = run_ssfm(tx, cfg['fiber'], cfg['ssfm'])\n",
    "rx_amp = apply_amplifiers(rx_raw, cfg['bands'])\n",
    "rx_dsp = receiver_dsp(rx_amp, cfg['dsp'], cfg['fiber'])\n",
    "\n",
    "# Compute Metrics\n",
    "metrics = compute_metrics(rx_dsp, tx, band_plan, cfg['metrics'])\n",
    "\n",
    "print(f\"Multi-band Aggregate Capacity: {metrics['aggregate_capacity_Tbs']:.2f} Tb/s\")\n",
    "print(f\"Mean GMI: {metrics['gmi_mean']:.2f} bits/symbol\")\n",
    "print(f\"Mean SNR: {metrics['snr_mean_dB']:.2f} dB\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot SNR across all 1,273 channels\n",
    "plt.figure(figsize=(12, 5))\n",
    "plt.plot(metrics['snr_per_channel_dB'], lw=1, label='Multi-band SNR')\n",
    "\n",
    "# Highlight band boundaries\n",
    "boundaries = np.cumsum([0] + [b['channels'] for b in cfg['bands'].values()])\n",
    "for i, band in enumerate(cfg['bands'].keys()):\n",
    "    plt.axvline(boundaries[i+1], color='k', ls='--', alpha=0.5)\n",
    "    plt.text((boundaries[i]+boundaries[i+1])/2, plt.ylim()[1]*0.95, band, ha='center')\n",
    "\n",
    "plt.title('SNR per Channel across O+E+S+C+L (No Neural NLI Cancellation)')\n",
    "plt.xlabel('Channel Index (1 to 1,273)')\n",
    "plt.ylabel('SNR (dB)')\n",
    "plt.grid(True, alpha=0.3)\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.10"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}